In [36]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(
    "/media/lazy/28481E65481E31D4/Placement Drive/AI ML Position/Dataset/test.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Filling null values
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

In [ ]:
# Replacing Male -> 0 and Female -> 1
df['Sex'] = df['Sex'].replace({'male': '0' , 'female': '1'})

In [ ]:
# Removing Ticket Column
df.drop('Ticket', axis=1, inplace=True)

In [ ]:
# Rounding up the Fare column data upto 2 decimal places
df['Fare'] = df['Fare'].round(2)

In [12]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [13]:
# 1. Cabin_Known
df['Cabin_Known'] = df['Cabin'].notna().astype(int)

# 2. Cabin_Deck
df['Cabin_Deck'] = df['Cabin'].fillna('Unknown').str[0]

# 3. One-hot encode Cabin_Deck if used in modeling
cabin_dummies = pd.get_dummies(df['Cabin_Deck'], prefix='Deck')
df = pd.concat([df, cabin_dummies], axis=1)

# 4. Drop the original Cabin column
df.drop('Cabin', axis=1, inplace=True)

In [14]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,...,Cabin_Deck,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.25,S,...,U,False,False,False,False,False,False,False,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.28,C,...,C,False,False,True,False,False,False,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.92,S,...,U,False,False,False,False,False,False,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.10,S,...,C,False,False,True,False,False,False,False,False,False
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.05,S,...,U,False,False,False,False,False,False,False,False,True


In [ ]:
# Finding Missing Values
missing_values = df.isnull().sum()
print(missing_values)

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       2
Cabin_Known    0
Cabin_Deck     0
Deck_A         0
Deck_B         0
Deck_C         0
Deck_D         0
Deck_E         0
Deck_F         0
Deck_G         0
Deck_T         0
Deck_U         0
dtype: int64


In [ ]:
# Adding new columns
embarked_mode = df['Embarked'].mode(
)[0] if 'Embarked' in df.columns and not df['Embarked'].mode().empty else "S"

In [ ]:
# Adding new columns
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [18]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,...,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.25,S,...,False,False,False,False,False,False,False,True,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.28,C,...,False,True,False,False,False,False,False,False,2,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.92,S,...,False,False,False,False,False,False,False,True,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.10,S,...,False,True,False,False,False,False,False,False,2,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.05,S,...,False,False,False,False,False,False,False,True,1,1


In [19]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked        object
Cabin_Known      int64
Cabin_Deck      object
Deck_A            bool
Deck_B            bool
Deck_C            bool
Deck_D            bool
Deck_E            bool
Deck_F            bool
Deck_G            bool
Deck_T            bool
Deck_U            bool
FamilySize       int64
IsAlone          int64
dtype: object

In [ ]:
# Changing datatype for column
dtype_mapping = {
    'PassengerId': 'int64',
    'Survived':   'int64',
    'Pclass':     'int64',
    'Name':       'object',
    'Sex':        'category',
    'Age':        'float64',
    'SibSp':      'int64',
    'Parch':      'int64',
    'Fare':       'float64',
    'Embarked':   'category',
    'Cabin_Known': 'int64',
    'Cabin_Deck': 'category',
    'Deck_A':     'bool',
    'Deck_B':     'bool',
    'Deck_C':     'bool',
    'Deck_D':     'bool',
    'Deck_E':     'bool',
    'Deck_F':     'bool',
    'Deck_G':     'bool',
    'Deck_T':     'bool',
    'Deck_U':     'bool',
    'FamilySize': 'int64',
    'IsAlone':    'int64'
}

for col, dtype in dtype_mapping.items():
    df[col] = df[col].astype(dtype)

print(df.dtypes)

PassengerId       int64
Survived          int64
Pclass            int64
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Fare            float64
Embarked       category
Cabin_Known       int64
Cabin_Deck     category
Deck_A             bool
Deck_B             bool
Deck_C             bool
Deck_D             bool
Deck_E             bool
Deck_F             bool
Deck_G             bool
Deck_T             bool
Deck_U             bool
FamilySize        int64
IsAlone           int64
dtype: object


In [21]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,...,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.25,S,...,False,False,False,False,False,False,False,True,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.28,C,...,False,True,False,False,False,False,False,False,2,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.92,S,...,False,False,False,False,False,False,False,True,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.10,S,...,False,True,False,False,False,False,False,False,2,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.05,S,...,False,False,False,False,False,False,False,True,1,1


In [23]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Fare', 'Embarked', 'Cabin_Known', 'Cabin_Deck', 'Deck_A',
       'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T',
       'Deck_U', 'FamilySize', 'IsAlone'],
      dtype='object')

In [25]:
df.drop('Cabin_Deck', axis=1, inplace=True)

In [37]:
df.to_csv("train_preprocessed.csv", index=False)